# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Adrián Cardona Young

**ID**: alc354

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\adric\OneDrive - Cornell University\Documents\3. Env Sys Analysis\HWs\hw5-alc354-hw5`


In [2]:
import Pkg; Pkg.add("CSV")

   Resolving package versions...
  No Changes to `C:\Users\adric\OneDrive - Cornell University\Documents\3. Env Sys Analysis\HWs\hw5-alc354-hw5\Project.toml`
  No Changes to `C:\Users\adric\OneDrive - Cornell University\Documents\3. Env Sys Analysis\HWs\hw5-alc354-hw5\Manifest.toml`


In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using CSV
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

City 1: \
$ W_1 = 100 $ Mg/day


City 2: \
$ W_2 = 90 $ Mg/day


City 3: \
$ W_3 = 120 $ Mg/day



In [55]:
cities = ["City 1", "City 2", "city 3"]
cat = ["Food Wastes", "Paper & Cardboard", "Plastics", "Textiles", "Rubber, Leather", "Wood", "Yard Wastes", "Glass", "Ferrous", "Aluminum", "Other Metal", "Miscellaneous"]
mass = [100, 90, 120]
pcnt_mass = [0.15, 0.40, 0.05, 0.03, 0.02, 0.05, 0.18, 0.04, 0.02, 0.02, 0.01, 0.03]
pcnt_ash = [0.08, 0.07, 0.05, 0.10, 0.15, 0.02, 0.02, 1, 1, 1, 1, 0.7]
pcnt_recyclable = [0, 0.55, 0.15, 0.10, 0, 0.30, 0.40, 0.60, 0.75, 0.80, 0.50, 0]

#Finding mass of each waste for each city
city1 = zeros(12)
city2 = zeros(12)
city3 = zeros(12)
for i in 1:12
    city1[i] = mass[1]*pcnt_mass[i]
    city2[i] = mass[2]*pcnt_mass[i]
    city3[i] = mass[3]*pcnt_mass[i]
end

#Mass recycled

rmass1 = 0
rmass2 = 0
rmass3 = 0

for i in 1:12
    rmass1 += city1[i]*pcnt_recyclable[i]
    rmass2 += city2[i]*pcnt_recyclable[i]
    rmass3 += city3[i]*pcnt_recyclable[i]
end

println("Recyclable mass for City 1 = ", round(rmass1; digits=2), " Mg/day")
println("Recyclable mass for City 2 = ", round(rmass2; digits=2), " Mg/day")
println("Recyclable mass for City 3 = ", round(rmass3; digits=2), " Mg/day")

println("-"^40)

#Ash Fraction
#Direct to incineration --> table value
#Recycling residuals sent to WTE --> non-recycled fraction * table fraction

#Percent of mass that is recycling residuals
pcnt_resid = zeros(12)
for i in 1:12
    pcnt_resid[i] = 1 - pcnt_recyclable[i]
end

res1 = zeros(12)
res2 = zeros(12)
res3 = zeros(12)

#Residual mass of each waste
for i in 1:12
    res1 = city1[i]*pcnt_resid
    res2 = city2[i]*pcnt_resid
    res3 = city3[i]*pcnt_resid
end

#Ash mass from recycling residuals for each city for each waste
ramass1 = 0
ramass2 = 0
ramass3 = 0

for i in 1:12
    ramass1 += city1[i]*pcnt_resid[i]*pcnt_ash[i]
    ramass2 += city2[i]*pcnt_resid[i]*pcnt_ash[i]
    ramass3 += city3[i]*pcnt_resid[i]*pcnt_ash[i]
end

println("Ash mass for City 1", round(ramass1; digits=1), " Mg/day")
println("Ash mass for City 2", round(ramass1; digits=1), " Mg/day")
println("Ash mass for City 3", round(ramass1; digits=1), " Mg/day")

Recyclable mass for City 1 = 37.75 Mg/day
Recyclable mass for City 2 = 33.98 Mg/day
Recyclable mass for City 3 = 45.3 Mg/day
----------------------------------------
Ash mass for City 18.6 Mg/day
Ash mass for City 28.6 Mg/day
Ash mass for City 38.6 Mg/day


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

I = {1, 2, 3} (cities) \
J = {1, 2, 3}, where 1 is WTE, 2 is MRF, and 3 is LF

$ W_{i,j} $ indicates waste stream from city i in I to waste disposal site j in J.

Decision variables: \
$ W_{1,1}, W_{1,2}, W_{1,3}, W_{2,1}, W_{2,2}, W_{2,3}, W_{3,1}, W_{3,2}, W_{3,3} $ waste streams from cities to disposal sites in Mg/day \
$ R_{1,3} $ residual amount sent from WTE to LF in Mg/day \
$ R_{2,3} $ residual amount sent from MRF to LF in Mg/day \
$ R_{2,1} $ residual amount sent from MRF to WTE in Mg/day \
$ Y_1, Y_2, Y_3 $ operational status of facility j in J (unitless)

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

Objective: Minimize cost

##### Transport costs

Transportation cost: $1.5/Mg-km

$ T = 1.5 \cdot \displaystyle \sum_{i \in I,j \in J} W_{i,j} \cdot l_{i,j} $ where $ W_{i,j} = $ waste transported from i to j and $ l_{i,j} = $ distance between i and j

##### Disposal costs

$ D = \displaystyle \sum_{j \in J} c_j \cdot Y_j + b_j \sum_{i \in I} W_{i,j} $ where $ c_j = $ fixed operating costs, $ b_j = $ variable disposal cost, and $ Y_j = $ indicator variable for facility operation

Objective min $ Z = \displaystyle \sum_{i \in I} \sum_{j \in J} 1.5 \cdot l_{i,j} \cdot W_{i,j} + \sum_{j \in J} (c_j \cdot Y_j + b_j \sum_{i \in I} W_{i,j}) $

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

LF, MRF, WTE

I = {1, 2, 3} (cities) \
J = {1, 2, 3}, where 1 is WTE, 2 is MRF, and 3 is LF

Non-exceedence of plant capacity: \
WTE: $ W_{1,1} + W_{2,1} + W_{3,1} + R_{2,1} \leq 210 $ \
MRF: $ W_{1,2} + W_{2,2} + W_{3,2} \leq 350 $ \
LF: $ W_{1,3} + W_{2,3} + W_{3,3} + R_{1,3} + R_{2,3} \leq 200 $

Treatment of all waste: \
$ W_{1,1} + W_{1,2} + W_{1,3} = 100 $ \
$ W_{2,1} + W_{2,2} + W_{2,3} = 90 $ \
$ W_{3,1} + W_{3,2} + W_{3,3} = 120 $

$Y_1 = 0 $ if $ W_{1,1} + W_{2,1} + W_{3,1} + R_{2,1} = 0$, else $ Y_1 = 1 $ \
$Y_2 = 0 $ if $ W_{1,2} + W_{2,2} + W_{3,2} = 0 $, else $ Y_2 = 1 $ \
$Y_3 = 1 $

Non-negativity: \
$ W_{i,j}, R_{j,j^*} \geq 0 $ where $j^*$ indicates a different j in J.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [19]:
cities = [1, 2, 3]
facilities = [:LF, :MRF, :WTE]

struct waste
    name::Symbol
    percent::Float64
    ash::Float64
    recycle::Float64
end

#Waste type, percentage of total waste, ash fraction, recycling fraction:
components = [waste(:Food, 0.15, 0.08, 0.00),
    waste(:Paper, 0.40, 0.07, 0.55),
    waste(:Plastics, 0.05, 0.05, 0.15),
    waste(:Textiles, 0.03, 0.10, 0.10),
    waste(:Rubber, 0.02, 0.15, 0.00),
    waste(:Wood, 0.05, 0.02, 0.30),
    waste(:Yard, 0.18, 0.02, 0.40),
    waste(:Glass, 0.04, 1.00, 0.60),
    waste(:Ferrous, 0.02, 1.00, 0.75),
    waste(:Aluminum, 0.02, 1.00, 0.80),
    waste(:OtherMetal, 0.01, 1.00, 0.50),
    waste(:Misc, 0.03, 0.70, 0.00)]

prod = Dict(1 => 100, 2 => 90, 3 => 120) #waste produced 
cap = Dict(:LF => 200, :MRF => 350, :WTE => 210) #treatment capacity
fc = Dict(:LF => 2000, :MRF => 1500, :WTE => 2500) #fixed costs
vc = Dict(:LF => 50, :MRF => 7, :WTE => 60) #variable costs
r_cost = 40 #$/Mg, recycling cost
t_cost = 1.5#$/Mg-km, transport cort
recycle_factor = sum(c.percent * c.recycle for c in components)

ash_factor = 0.0
for c in components
    ash_factor += c.percent * c.ash
end

dist = Dict(
    (1, :LF) => 5,
    (1, :MRF) => 30,
    (1, :WTE) => 15,
    (2, :LF) => 15,
    (2, :MRF) => 25,
    (2, :WTE) => 10,
    (3, :LF) => 13,
    (3, :MRF) => 45,
    (3, :WTE) => 20,
    (:MRF, :LF) => 32,
    (:MRF, :WTE) => 15,
    (:WTE, :LF) => 18,
)

#Initiating model

model = Model(HiGHS.Optimizer)
set_silent(model)

#Indicator
@variable(model, Y[1:3], Bin)
@constraint(model, force_LF, Y[3] == 1)

# 1 is WTE, 2 is MRF, and 3 is LF

# Waste Streams:
# City i to facility f
@variable(model, 0 <= stream[i in cities, f in facilities])

# MRF residuals to LF/WTE
@variable(model, 0 <= R23) #to LF
@variable(model, 0 <= R21) #to WTE

# WTE ash to LF
@variable(model, 0 <= R13)

# Recycled material from MRF
@variable(model, 0 <= r_tot)

# Mass balance helper variables
@variable(model, 0 <= mrf_in)  # Total waste entering MRF
@variable(model, 0 <= mrf_res)  # Non-recycled portion from MRF
@variable(model, 0 <= wte_in)   # Total waste entering WTE


###Constraints

for i in cities
    @constraint(model, sum(stream[i, f] for f in facilities) == prod[i])
end

#### Facilities Cap
# WTE capacity (Y[1] = WTE)
@constraint(model, sum(stream[i, :WTE] for i in cities) + R21 <= cap[:WTE] * Y[1])

# MRF capacity (Y[2] = MRF)
@constraint(model, sum(stream[i, :MRF] for i in cities) <= cap[:MRF] * Y[2])

# LF Capacity (Y[3] = LF)
@constraint(model, sum(stream[i, :LF] for i in cities) + R23 + R13 <= cap[:LF] * Y[3])

##### MRF Mass Balance
#Total waste entering MRF
@constraint(model, mrf_in == sum(stream[i, :MRF] for i in cities))

# Recycling amount (using AVERAGE recycling rate from table)
@constraint(model, r_tot == mrf_in * recycle_factor)

# MRF residuals (non-recycled portion)
@constraint(model, mrf_res == mrf_in - r_tot)

# MRF residuals must go to LF or WTE
@constraint(model, mrf_res == R23 + R21)

#### WTE Mass Balance
# Total waste entering WTE
@constraint(model, wte_in == sum(stream[i, :WTE] for i in cities) + R21)

# Ash from WTE goes to landfill
@constraint(model, R13 == wte_in * ash_factor)


### Developing Objective Function

# Transportation costs
# City to facility transport
t_city = t_cost * sum(stream[i, f] * dist[(i, f)] for i in cities, f in facilities)
t_mrf = t_cost * (R23 * dist[(:MRF, :LF)] + R21 * dist[(:MRF, :WTE)])
t_wte = t_cost * R13 * dist[(:WTE, :LF)]

tc_tot = t_city + t_mrf + t_wte

# Tipping fees
tipc_tot = (
    sum(stream[i, :LF] for i in cities) * vc[:LF] +
    sum(stream[i, :MRF] for i in cities) * vc[:MRF] +
    sum(stream[i, :WTE] for i in cities) * vc[:WTE] +
    R23 * vc[:LF] + R21 * vc[:WTE] + R13 * vc[:LF])

# Recycling cost at MRF
rc_tot = r_tot * r_cost

# Fixed costs
fc_tot = fc[:LF] * Y[3] + fc[:MRF] * Y[2] + fc[:WTE] * Y[1]

# Total cost minimization
@objective(model, Min, fc_tot + tipc_tot + rc_tot + tc_tot)

2000 Y[3] + 1500 Y[2] + 2500 Y[1] + 57.5 stream[1,LF] + 72.5 stream[2,LF] + 69.5 stream[3,LF] + 52 stream[1,MRF] + 44.5 stream[2,MRF] + 74.5 stream[3,MRF] + 82.5 stream[1,WTE] + 75 stream[2,WTE] + 90 stream[3,WTE] + 98 R23 + 82.5 R21 + 77 R13 + 40 r_tot

In [17]:
optimize!(model)

total_cost = objective_value(model)
println("Optimal Objective Value: \$", Int(round(total_cost)))

Optimal Objective Value: $27855


In [31]:
# Check facility decisions
println("\nFacility Use (Y variables):")
println("WTE: ", value(Y[1]))
println("MRF: ", value(Y[2]))
println("LF: ", value(Y[3]))

# Check waste flows
println("\nWaste flows (Mg/day):")
total_waste_flow = 0
for i in cities
    println("City $i:")
    for f in facilities
        flow = value(stream[i, f])
        println("   to $f: $(round(flow, digits=2)) Mg")
        total_waste_flow += flow
    end
end


Facility Use (Y variables):
WTE: 1.0
MRF: -0.0
LF: 1.0

Waste flows (Mg/day):
City 1:
   to LF: 100.0 Mg
   to MRF: 0.0 Mg
   to WTE: 0.0 Mg
City 2:
   to LF: -0.0 Mg
   to MRF: -0.0 Mg
   to WTE: 90.0 Mg
City 3:
   to LF: 78.41 Mg
   to MRF: 0.0 Mg
   to WTE: 41.59 Mg


#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is 1500 MW.
In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

In [12]:
gens = DataFrame(CSV.File("data/generators.csv"))

Row,Plant,Resource,Pmin,Pmax,VarCost,Ramp
,String15,String15,Int64,Int64,Int64,Int64
1,Biomass,Biomass,0,100,5,100
2,Hydroelectric,Hydroelectric,0,500,0,500
3,Geothermal,Geothermal,0,400,0,400
4,NG CCGT,NG CCGT,220,500,23,100
5,NG CT,NG CT,100,250,38,200
6,Wind,Wind,0,300,0,300
7,Solar,Solar,0,500,0,500


G = geothermal \
H = hydroelectric \
C = coal \
CCGT = natural gas combined cyclone gas turbine \
CT = natural gas combustion turbine \
W = wind \
S = solar

Four scenarios: \
1: $ d^{1}_2 = 1200 $ MW, $c_s = 0.95, c_w = 0.4 $ \
2: $ d^{2}_2 = 1200 $ MW, $c_s = 0.75, c_w = 0.5 $ \
3: $ d^{3}_2 = 1500 $ MW, $c_s = 0.95, c_w = 0.4 $ \
4: $ d^{4}_2 = 1500 $ MW, $c_s = 0.75, c_w = 0.5 $ \
For the values d, the subscript indicates the time period and the superscript indicates the scenario.

Variable cost: $VC_g = $ variable cost of production for generator g in G \
Ramping constraints: $ R_g = $ ramping limit between time periods for generator g in G \
Capacity factor: $c^{s}_{t} = $ capacity factor for scenario s and time period

Calculating probabilities of each scenario: \
1: $ \pi_1 = 0.75 \cdot 0.7 = 0.525 $ \
2: $ \pi_2 = 0.75 \cdot 0.3 = 0.225 $ \
3: $ \pi_3 = 0.25 \cdot 0.7 = 0.175 $\
4: $ \pi_4 = 0.25 \cdot 0.3 = 0.075 $

G = {G, H, C, CCGT, CT, W, S}, generators \
T = {1, 2}, time periods \
S = {1, 2, 3, 4}, scenarios

Decision variables: \
$ y^{s}_{g,t} = $ power output from generator g in G at time t in T for scenario s in S

Objective: minimize Z: \
$ Z = \displaystyle \sum_{g \in G} VC_g \cdot y_{g,1} \cdot c^{s}_{1} + \sum_{s \in S} \pi_s \sum_{g \in G} VC_g \cdot y^{s}_{g,2} $ \

Constraints:

Period 1: \
$ \displaystyle \sum_{g \in G} y_{g,1} \leq 1100 $

Period 2: \
$ \displaystyle \sum_{g \in G} y^{s}_{g,2} \leq d^{s} $, where $ d^1 = d^2 = 1200 $ and $d^3 = d^4 = 1500 $

General: \
$ P_{min, g} \leq y^{s}_{g,t} \cdot c^{s}_{t} \leq P_{max, g} $ \
$ -R_g \leq y^{s}_{g,2} - y^{s}_{g,1} \leq R_g $ \
$ y^{s}_{g,t} \geq 0 $

Calculating maximum solar and wind outputs for period 1: \
Solar: $ 500 \cdot 0.9 = 450 $ MW \
Wind: $ 300 \cdot 0.45 = 135 $ MW

Calculating maximum solar and wind outputs based on each scenario for period 2: \
1 and 3: $ 500 \cdot 0.95 = 475 $ MW solar, $ 300 \cdot 0.4 =  120 $ MW wind \
2 and 4: $ 500 \cdot 0.75 = 375 $ MW solar, $ 300 \cdot 0.5 =  150 $ MW wind 

## References

List any external references consulted, including classmates.